# Scrapping

Scrap the web and save content on local as json files

In [4]:
import time
import os
import re
import json

import requests
from bs4 import BeautifulSoup, Tag
from urllib.parse import urlparse
from datetime import datetime
from pprint import pprint
import numpy as np

from tqdm import tqdm

In [ ]:
def scrape_page(id, url):
    global pages_scraped
    url = url.split("#")[0]
    
    if url in visited or pages_scraped >= MAX_PAGES:
        return
        
    try:
        print(f"{pages_scraped+1}: {url}")
        visited.add(url)

        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, "lxml")

        page_obj = {
            "url": url,
            'xml':str(soup),
        }
        pages_scraped += 1

        with open(os.path.join(DATA_DIR, f"{id}_{pages_scraped}.json"), "w", encoding="utf-8") as f:
            json.dump(page_obj, f, ensure_ascii=False, indent=2)
            
        for link in soup.find_all("a", href=True):
            href = link["href"]
            start_with = url.replace(BASE_URL, "").replace(".html","")
            lang = link.get("lang", "en")
            if lang.startswith("en"):
                if href.startswith(start_with):
                    full_url = BASE_URL + href
                    if full_url not in visited:
                        scrape_page(id, full_url)
    except Exception as e:
        print("Error scraping", url, ":", e)

In [ ]:
BASE_URL = "https://www.canada.ca"
MAX_PAGES = 10000
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
BASE_URL = "https://www.canada.ca"

urls = {
    'visit-canada' : 'https://www.canada.ca/en/immigration-refugees-citizenship/services/visit-canada.html',
    'immigrate-canada' : 'https://www.canada.ca/en/immigration-refugees-citizenship/services/immigrate-canada.html',
    'study-canada' : 'https://www.canada.ca/en/immigration-refugees-citizenship/services/study-canada.html',
    'canadian-citizenship' : 'https://www.canada.ca/en/immigration-refugees-citizenship/services/canadian-citizenship.html',
    'permanent-residents' : 'https://www.canada.ca/en/immigration-refugees-citizenship/services/permanent-residents.html',
    'canadian-passports' : 'https://www.canada.ca/en/immigration-refugees-citizenship/services/canadian-passports.html',
    'settle-canada' : 'https://www.canada.ca/en/immigration-refugees-citizenship/services/settle-canada.html',
    'adopt-child-abroad' : 'https://www.canada.ca/en/immigration-refugees-citizenship/services/canadians/adopt-child-abroad.html'
}

for id, url in urls.items():
    pages_scraped = 0
    visited = set()
    scrape_page(id, url)

# Load downloaded json files

In [1]:
from pathlib import Path

In [2]:
directory_path = Path('data') 
files = [p.name for p in directory_path.iterdir() if p.is_file()]

In [5]:
pages = {}
for i in files:
    with open(f'data/{i}', "r") as f:
        pages[i] = json.load(f)

# extract data

In [6]:
def clean_html(soup: BeautifulSoup):
    # Remove unwanted tags
    for tag in soup(["script", "style", "nav", "header", "footer", "noscript"]):
        tag.decompose()

    # Remove "On this page" section
    for h2 in soup.find_all("h2"):
        if "On this page" in h2.get_text():
            parent = h2.find_parent()
            if parent:
                parent.decompose()

    return soup

In [7]:
def table_to_markdown(table: Tag) -> str:
    rows = []
    headers = []

    # Extract headers
    header_row = table.find("thead")
    if header_row:
        headers = [
            th.get_text(strip=True).replace("\xa0", " ")
            for th in header_row.find_all("th")
        ]
        rows.append("| " + " | ".join(headers) + " |")
        rows.append("| " + " | ".join(["---"] * len(headers)) + " |")

    # Extract body rows
    for tr in table.find_all("tr"):
        cols = tr.find_all(["td", "th"])
        row = [col.get_text(strip=True).replace("\xa0", " ") for col in cols]
        if row:
            rows.append("| " + " | ".join(row) + " |")

    return "\n".join(rows)

In [8]:
def extract_meta(soup):
    meta = {}

    for tag in soup.find_all("meta"):
        name = tag.get("name")
        content = tag.get("content")

        if name and content:
            meta[name.lower()] = content

    return meta

def extract_breadcrumb(soup):
    breadcrumb = []
    nav = soup.find("nav", {"aria-label": "Breadcrumb"})
    
    if nav:
        for li in nav.find_all("li"):
            text = li.get_text(strip=True)
            if text:
                breadcrumb.append(text)

    return breadcrumb

    
def build_metadata(m_dict, section_title):
    m = m_dict.copy()
    m['section_title'] = section_title
    return m

In [9]:
def extract_documents(id, url, html_string=None):
    # response = requests.get(url)
    # soup = BeautifulSoup(response.text, "lxml")
    soup = BeautifulSoup(html_string, 'html.parser')
    
    meta_tags = extract_meta(soup)
    breadcrumb = extract_breadcrumb(soup)
    
    program = breadcrumb[-2] if len(breadcrumb) >= 2 else None
    topic = breadcrumb[-1] if breadcrumb else None
    
    page_title = soup.title.get_text(strip=True)
    parsed_url = urlparse(url)

    base_meta_data = {
        # "doc_id": id,
        "url": url,
        "path": parsed_url.path,

        "date": meta_tags.get("dcterms.modified"),
        "issued_date": meta_tags.get("dcterms.issued"),
        "language": meta_tags.get("dcterms.language"),

        "program": program,
        "topic": topic,
        "tags": meta_tags.get("dcterms.subject"),
        "metadata_subject": meta_tags.get("dcterms.subject"),

        "page_title": page_title,
        "metadata_title": meta_tags.get("dcterms.title"),

        "section_title": None,
        "description": meta_tags.get("description"),
        "keywords": meta_tags.get("keywords"),

        "archived": 'ARCHIVED'.lower() in page_title.lower()
    }

    main = soup.find("main")
    main = clean_html(main)

    documents = []

    h1 = main.find("h1")
    intro_content = []
    capture = False
    
    for element in main.descendants:
    
        if isinstance(element, Tag):
    
            # When we hit H1 → start capturing
            if element.name == "h1":
                capture = True
                continue
    
            # Stop when first H2 appears
            if element.name == "h2":
                break
    
            if capture:
                if element.name == "table":
                    intro_content.append(table_to_markdown(element))
    
                elif element.name in ["p", "li"]:
                    text = element.get_text(" ", strip=True)
                    if text:
                        intro_content.append(text)
    if intro_content:
        documents.append({
            "content": "\n".join(intro_content).strip(),
            "metadata": build_metadata(base_meta_data, h1.get_text(strip=True))
        })
        

    # ---- STEP 2: Capture H2/H3 Sections ----
    current_section = None
    current_content = []

    for element in main.find_all(["h2", "h3", "h4", "p", "li", "table"]):

        if element.name in ["h2", "h3", "h4"]:
            # Save previous section
            if current_section and current_content:
                documents.append({
                    "content": "\n".join(current_content).strip(),
                    "metadata": build_metadata(base_meta_data, current_section)
                })

            current_section = element.get_text(strip=True)
            current_content = []

        elif element.name == "table":
            current_content.append(table_to_markdown(element))

        elif element.name in ["p", "li"]:
            text = element.get_text(" ", strip=True)
            if text:
                current_content.append(text)

    # Save last section
    if current_section and current_content:
        documents.append({
            "content": "\n".join(current_content).strip(),
            "metadata": build_metadata(base_meta_data, current_section)
        })
    # final_docs = []
    for index, doc in enumerate(documents):
        doc['metadata']['doc_id'] = f"{id.replace('.json','')}_{str(index+1)}"
        
    return documents

In [10]:
documents_all_nested = []

for id, page in tqdm(pages.items()):
    url = page['url']
    html_string = page['xml']
    documents_all_nested.append(extract_documents(id, url, html_string=html_string))


100%|████████████████████████████████████████████████████████████████████████████████| 560/560 [00:04<00:00, 129.15it/s]


In [11]:
documents_all = [doc for nested_doc in  documents_all_nested for doc in nested_doc]

In [12]:
print(len(documents_all_nested))
print(len(documents_all))

560
5255


In [13]:
documents_all[0]

{'content': '1. About the process\n2. Who can apply\n3. How to apply\n4. After you apply',
 'metadata': {'url': 'https://www.canada.ca/en/immigration-refugees-citizenship/services/canadian-citizenship/resume-canadian-citizenship/after-resuming-next-steps.html',
  'path': '/en/immigration-refugees-citizenship/services/canadian-citizenship/resume-canadian-citizenship/after-resuming-next-steps.html',
  'date': '2024-01-09',
  'issued_date': '2007-03-31',
  'language': 'eng',
  'program': None,
  'topic': None,
  'tags': 'Citizenship; Canadian citizen; Resumption of citizenship',
  'metadata_subject': 'Citizenship; Canadian citizen; Resumption of citizenship',
  'page_title': 'Resume Canadian citizenship: After you apply - Canada.ca',
  'metadata_title': 'Resume Canadian citizenship: After you apply',
  'section_title': 'Resume Canadian citizenship: After you apply',
  'description': 'Resume Canadian citizenship: After you apply',
  'keywords': 'Citizenship; Canadian citizen; Resumption of

In [ ]:
# def build_embedding_text(doc):
#     return f"""
# Title: {doc['metadata']['metadata_title']}
# Description: {doc['metadata']['description']}
# Keywords: {doc['metadata']['keywords']}

# Section: {doc['metadata']['section_title']}
# Content:
# {doc['content']}
# """.strip()

# Filtering


In [33]:
import os
import re
import numpy as np

In [1]:
import pandas as pd
import json
import statistics
from transformers import AutoTokenizer
from pathlib import Path
# pd.set_option('display.max_colwidth', None)
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.width', None)

/Users/ankit/Projects/learn_llm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.DataFrame(documents_all)
df.head(1)

,content,metadata,doc_id,language,url,topic,tags,metadata_subject,page_title,metadata_title,...,description,keywords,archived,date,issued_date,path,char_counts,word_counts,token_counts,embedding_content
index,,,,,,,,,,,,,,,,,,,,,
36864,See the interactive community boundary map,{'url': 'https://www.canada.ca/en/immigration-...,immigrate-canada_67_4,eng,https://www.canada.ca/en/immigration-refugees-...,None,Information and Communications;Information dis...,Information and Communications;Information dis...,Rural and Francophone Community Immigration pi...,Rural and Francophone Community Immigration pi...,...,How candidates and employers can participate i...,"francophone immigration, rural immigration, rc...",False,2025-09-26,2025-04-07,/en/immigration-refugees-citizenship/services/...,42,6,6,Title: Rural and Francophone Community Immigra...


In [17]:
len(df)

5030

In [18]:
def clean_content(text):
    # Remove URLs from content
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)

    # Replace multiple spaces/newlines with single space
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [21]:
df['doc_id'] = df['metadata'].apply(lambda x: x['doc_id'])
df['language'] = df['metadata'].apply(lambda x: x['language'])
df['url'] = df['metadata'].apply(lambda x: x['url'])
df['topic'] = df['metadata'].apply(lambda x: x['topic'])
df['tags'] = df['metadata'].apply(lambda x: x['tags'])
df['metadata_subject'] = df['metadata'].apply(lambda x: x['metadata_subject'])
df['page_title'] = df['metadata'].apply(lambda x: x['page_title'])
df['metadata_title'] = df['metadata'].apply(lambda x: x['metadata_title'])
df['section_title'] = df['metadata'].apply(lambda x: x['section_title'])
df['description'] = df['metadata'].apply(lambda x: x['description'])
df['keywords'] = df['metadata'].apply(lambda x: x['keywords'])
df['archived'] = df['metadata'].apply(lambda x: x['archived'])
df['date'] = df['metadata'].apply(lambda x: x['date'])
df['issued_date'] = df['metadata'].apply(lambda x: x['issued_date'])
df['path'] = df['metadata'].apply(lambda x: x['path'])
df['content'] = df['content'].apply(lambda x: clean_content(x))
df['char_counts'] = df['content'].apply(lambda x: len(x))
df['word_counts'] = df['content'].apply(lambda x: len(x.split()))

In [22]:
df['language'].value_counts()

language
eng    5030
Name: count, dtype: int64

In [23]:
df['archived'].value_counts()

archived
False    5030
Name: count, dtype: int64

In [24]:
pd.DataFrame(df['section_title'].value_counts()).head()

,count
section_title,
Study permit application: Option 1 of 2,128
PGWP application: Option 2 of 2,128
How to apply,31
3. Find a guarantor,27
2. Get all the required documents and the child’s passport photo,26


In [25]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-0.5B")
df['token_counts'] = df['content'].apply(lambda x: len(tokenizer.encode(x)))

Token indices sequence length is longer than the specified maximum sequence length for this model (69398 > 32768). Running this sequence through the model will result in indexing errors


In [26]:
df[['char_counts', 'word_counts', 'token_counts']].describe().astype(int)

,char_counts,word_counts,token_counts
count,5030,5030,5030
mean,628,107,141
std,4455,758,1094
min,26,6,6
25%,123,20,38
50%,235,41,49
75%,533,92,114
max,282433,48472,69398


In [27]:
df = df.sort_values('token_counts')

In [28]:
df = df[(df['word_counts']>5)]
df = df[(df['token_counts']>5)]
df = df[~(df['archived'])]
df = df[(df['section_title'])!='Footnotes']

In [29]:
len(df)

5030

In [30]:
df['archived'].value_counts()

archived
False    5030
Name: count, dtype: int64

In [31]:
df[['char_counts', 'word_counts', 'token_counts']].describe().astype(int)

,char_counts,word_counts,token_counts
count,5030,5030,5030
mean,628,107,141
std,4455,758,1094
min,26,6,6
25%,123,20,38
50%,235,41,49
75%,533,92,114
max,282433,48472,69398


In [34]:
df['token_counts'].quantile(np.arange(0,1.05,0.05))

0.00        6.0
0.05       14.0
0.10       20.0
0.15       23.0
0.20       29.0
0.25       38.0
0.30       41.0
0.35       43.0
0.40       44.0
0.45       47.0
0.50       49.0
0.55       57.0
0.60       71.0
0.65       86.0
0.70      101.0
0.75      114.0
0.80      137.0
0.85      180.0
0.90      252.0
0.95      438.0
1.00    69398.0
Name: token_counts, dtype: float64

In [35]:
def build_embedding_text(content, metadata):
    return f"""
Title: {metadata['metadata_title']}
Description: {metadata['description']}
Keywords: {metadata['keywords']}
Section: {metadata['section_title']}
Content: {content}
""".strip()

In [36]:
df['embedding_content'] = df.apply(lambda x: build_embedding_text(x['content'], x['metadata']), axis=1)

In [37]:
deduped_df = df.drop_duplicates('content')

In [38]:
len(deduped_df)

4010

### Analysis

In [ ]:
a = pd.merge(df, deduped_df[['doc_id','embedding_content']], on='embedding_content', how='left')

In [ ]:
b = a[a['doc_id_x']!=a['doc_id_y']]

In [ ]:
len(b)

In [ ]:
b = b.sort_values('content')

In [ ]:
# df[df['content']=='After we confirm your status, you’ll get an automated email when your electronic confirmation of permanent residence (e-COPR) is ready. This document is proof of your new status in Canada. It can take up to a few weeks for your e-COPR to be processed. In the portal, we’ll also ask you for a photo so we can start making your first permanent resident (PR) card. You don’t need to apply for your first PR card. Once your photo is approved, we’ll send your PR card to the mailing address in Canada you gave us.']

In [ ]:
# b['content'].unique()

In [ ]:
df.to_parquet('data.parquet')

In [ ]:
# print(df[df['token_counts']>100000].iloc[0]['content'])

# Embbeding 

In [16]:
import pandas as pd
df = pd.read_parquet('data.parquet')

In [39]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [ ]:
# df.drop_duplicates('content')

In [40]:
len(deduped_df)

4010

In [41]:
len(deduped_df.columns)

21

In [42]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=300
)

documents = []

# for i, (content, metadata, _,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,) in df.iterrows():
for i, (content, metadata, _,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_,_) in deduped_df.iterrows():

    chunks = splitter.split_text(content)

    for id, chunk in enumerate(chunks):
        metadata['doc_id'] = f"{metadata['doc_id']}_{id}"
        doc = Document(
            page_content=build_embedding_text(chunk, metadata),
            metadata=metadata
        )
        documents.append(doc)
print(f"Total Documnets: {len(deduped_df)}")
print(f"Total chunked documents: {len(documents)}")

Total Documnets: 4010
Total chunked documents: 4597


In [57]:
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")
persist_directory = "vector_db/ircc"

In [ ]:
# vector_store = Chroma.from_documents(
#     documents,
#     embedding=embeddings,
#     persist_directory=persist_directory
# )
# vector_store.persist()
# print(f"Vector DB persisted to '{persist_directory}' directory.")

In [42]:
from tqdm import tqdm
from langchain_community.vectorstores import Chroma

batch_size = 500

for i in tqdm(range(0, len(documents), batch_size), desc="Indexing batches"):
    batch_docs = documents[i:i + batch_size]

    Chroma.from_documents(
        batch_docs,
        embedding=embeddings,
        persist_directory=persist_directory
    )

Indexing batches: 100%|█████████████████████████████████████████████████████████████████| 10/10 [06:40<00:00, 40.07s/it]


In [58]:
print("Loading existing vector DB...")

vector_store = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings
)

print("Vector DB loaded successfully.")

Loading existing vector DB...


/var/folders/ng/dp6yvncx7pl1qbny9711g9hr0000gn/T/ipykernel_26715/2889421506.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


Vector DB loaded successfully.


In [70]:
len(documents[0])

1

In [76]:
from tqdm import tqdm


In [79]:
BATCH_SIZE = 1

documents = []
metadatas = []
embeddings = []
ids = []

for offset in tqdm(range(0, vector_store._collection.count(), BATCH_SIZE)):

    batch = vector_store._collection.get(
        limit=BATCH_SIZE,
        offset=offset,
        include=["documents", "metadatas"]
    )
    d_dict = {'doc_id':batch['metadatas'][0]['doc_id'], 
              'page_content': batch["documents"][0], 
              'metadata': batch["metadatas"][0]}
    try:
        with open(os.path.join("documents", f"{d_dict['doc_id']}.json"), "w", encoding="utf-8") as f:
                json.dump(d_dict, f, ensure_ascii=False, indent=2)
    except:
        pass

    # documents.append(batch["documents"])
    # metadatas.append(batch["metadatas"])
    # embeddings.append(batch["embeddings"])
    # ids.append(batch["ids"])
    


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4597/4597 [00:08<00:00, 560.23it/s]


In [81]:
write me a prompt for openclaw agent:


i want it to create me a golden test set for RAG testing. 
i have documents stored in ~/project/ircc_rag_agent/notebook/documents as a json file. 
each json file has doc_id, metadata and page_content. 

output should be in this structure

{
  "query_id": "generate random id",
  "query": "create a meaning full query based on page_content",
  "expected_documents": [
    {
      "doc_id": "crs_age_points_table",
      "page_content": "Age – CRS Points",
      "metadata": {}
    }
  ]
}

it should acheive the task by spaning a sub agent.
sub agent should write a fina output as a json file in ~/project/ircc_rag_agent/notebook/test_set
maximum 300 docuemnts.

SyntaxError: invalid syntax (1902505525.py, line 1)